# 🌿 Notebook 1 — Data Preparation (Kaggle version)
**Crop Disease Diagnostician** | PlantVillage via Kaggle

TFDS PlantVillage is broken (Mendeley 403). This version uses Kaggle directly.

## Before running:
1. Create a free account at [kaggle.com](https://kaggle.com)
2. Go to: **kaggle.com → your profile icon → Settings → API → Create New Token**
3. It downloads a file called `kaggle.json` to your computer
4. Come back here and run the cells — you'll be asked to upload that file

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!pip install -q kaggle

In [ ]:
import os, json, shutil, random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: SET UP KAGGLE CREDENTIALS
# Upload your kaggle.json when prompted
# ─────────────────────────────────────────────────────────────────────────────

print('Please upload your kaggle.json file when prompted below:')
uploaded = files.upload()   # A file picker dialog will appear

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle credentials configured!')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: DOWNLOAD PLANTVILLAGE FROM KAGGLE
# Dataset: abdallahalidev/plantvillage-dataset (~1 GB, takes ~3-5 min)
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs('/content/kaggle_download', exist_ok=True)

print('Downloading PlantVillage dataset from Kaggle...')
!kaggle datasets download -d abdallahalidev/plantvillage-dataset \
    -p /content/kaggle_download --unzip

print('\nDownload complete. Finding image folder...')

# The dataset extracts to: 'plantvillage dataset/color/'
# Walk the download dir to find the folder containing class subfolders
COLOR_DIR = None
for root, dirs, _ in os.walk('/content/kaggle_download'):
    # Look for a folder that contains our expected class names
    if any('Tomato' in d for d in dirs) and any('Corn' in d for d in dirs):
        COLOR_DIR = root
        break

if COLOR_DIR is None:
    # Fallback: try common paths
    for candidate in [
        '/content/kaggle_download/plantvillage dataset/color',
        '/content/kaggle_download/color',
        '/content/kaggle_download/PlantVillage',
    ]:
        if os.path.exists(candidate):
            COLOR_DIR = candidate
            break

assert COLOR_DIR is not None, 'Could not find image folder! Run: !find /content/kaggle_download -type d | head -30'

all_classes = sorted(os.listdir(COLOR_DIR))
print(f'\n✅ Found dataset at: {COLOR_DIR}')
print(f'Total classes available: {len(all_classes)}')
for c in all_classes:
    n = len(os.listdir(os.path.join(COLOR_DIR, c)))
    print(f'  {c:<55} {n:>5} images')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TARGET CLASSES
# Kaggle folder name → our internal class key
# ─────────────────────────────────────────────────────────────────────────────

TARGET_CLASSES = {
    # Kaggle/PlantVillage folder name                        → our class key
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot':    'corn_gray_leaf_spot',
    'Corn_(maize)___Common_rust_':                           'corn_common_rust',
    'Corn_(maize)___healthy':                                'corn_healthy',
    'Potato___Early_blight':                                 'potato_early_blight',
    'Potato___Late_blight':                                  'potato_late_blight',
    'Potato___healthy':                                      'potato_healthy',
    'Tomato___Bacterial_spot':                               'tomato_bacterial_spot',
    'Tomato___Early_blight':                                 'tomato_early_blight',
    'Tomato___Late_blight':                                  'tomato_late_blight',
    'Tomato___Leaf_Mold':                                    'tomato_leaf_mold',
    'Tomato___healthy':                                      'tomato_healthy',
}

# Sorted alphabetically → deterministic 0..10 index assignment
LABELS_MAP = sorted(TARGET_CLASSES.values())
our_key_to_idx = {key: idx for idx, key in enumerate(LABELS_MAP)}
NUM_CLASSES = len(LABELS_MAP)

print(f'Our {NUM_CLASSES} target classes (model output index → class key):')
for idx, key in enumerate(LABELS_MAP):
    print(f'  [{idx}] {key}')

# Verify all target folders exist in the downloaded dataset
print('\nVerifying folders exist:')
missing = []
for folder_name in TARGET_CLASSES:
    path = os.path.join(COLOR_DIR, folder_name)
    exists = os.path.exists(path)
    status = '✅' if exists else '❌ MISSING'
    print(f'  {status}  {folder_name}')
    if not exists:
        missing.append(folder_name)

if missing:
    print(f'\n⚠️  {len(missing)} folder(s) not found!')
    print('Run: !ls "{COLOR_DIR}" | grep -i corn   to check exact folder names')
else:
    print('\n✅ All target folders found!')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ORGANIZE INTO TRAIN / VAL / TEST DIRECTORIES (80 / 10 / 10)
#
# We copy (not move) images into:
#   /content/data/train/<class_key>/
#   /content/data/val/<class_key>/
#   /content/data/test/<class_key>/
#
# Using our class key (e.g. 'tomato_early_blight') as the folder name
# so tf.keras.utils.image_dataset_from_directory assigns consistent labels.
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR = '/content/data'
SEED = 42
random.seed(SEED)

split_stats = {}
total_train = total_val = total_test = 0

for folder_name, class_key in TARGET_CLASSES.items():
    src_dir = os.path.join(COLOR_DIR, folder_name)
    images  = [f for f in os.listdir(src_dir)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)

    n       = len(images)
    n_val   = max(1, int(n * 0.10))
    n_test  = max(1, int(n * 0.10))
    n_train = n - n_val - n_test

    splits = {
        'train': images[:n_train],
        'val':   images[n_train:n_train + n_val],
        'test':  images[n_train + n_val:],
    }

    for split_name, split_images in splits.items():
        dest = os.path.join(DATA_DIR, split_name, class_key)
        os.makedirs(dest, exist_ok=True)
        for img_file in split_images:
            shutil.copy(os.path.join(src_dir, img_file),
                        os.path.join(dest, img_file))

    split_stats[class_key] = {'train': n_train, 'val': n_val, 'test': n_test, 'total': n}
    total_train += n_train; total_val += n_val; total_test += n_test
    print(f'  {class_key:<30}  total={n:>4}  train={n_train:>4}  val={n_val:>3}  test={n_test:>3}')

print(f'\nTotals → train: {total_train}  val: {total_val}  test: {total_test}')
print(f'Data organised in: {DATA_DIR}/')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SAVE LABELS + SPLIT INFO
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs('/content/model', exist_ok=True)

# labels.json: list where index → class key (e.g. labels[0] = 'corn_gray_leaf_spot')
# IMPORTANT: this must be sorted alphabetically to match image_dataset_from_directory
with open('/content/model/labels.json', 'w') as f:
    json.dump(LABELS_MAP, f, indent=2)
print('Saved: /content/model/labels.json  ← copy this to app/public/model/')

split_info = {
    'total': total_train + total_val + total_test,
    'train': total_train, 'val': total_val, 'test': total_test,
    'num_classes': NUM_CLASSES,
    'labels': LABELS_MAP,
    'data_dir': DATA_DIR,
    'seed': SEED,
    'per_class': split_stats,
}
with open('/content/model/split_info.json', 'w') as f:
    json.dump(split_info, f, indent=2)
print('Saved: /content/model/split_info.json')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VISUALISE SAMPLE IMAGES
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

for i, class_key in enumerate(LABELS_MAP[:12]):
    ax = axes[i]
    class_dir = os.path.join(DATA_DIR, 'train', class_key)
    sample_img = os.path.join(class_dir, os.listdir(class_dir)[0])
    img = plt.imread(sample_img)
    ax.imshow(img)
    ax.set_title(class_key.replace('_', '\n'), fontsize=7)
    ax.axis('off')

plt.suptitle('Sample Images — Training Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/model/sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: /content/model/sample_images.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CLASS DISTRIBUTION BAR CHART
# ─────────────────────────────────────────────────────────────────────────────

class_counts = [split_stats[k]['train'] for k in LABELS_MAP]

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.barh(LABELS_MAP, class_counts, color='#16a34a')
ax.set_xlabel('Training images')
ax.set_title('Class Distribution — Training Set')
for bar, count in zip(bars, class_counts):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/model/class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

# Compute class weights for imbalance handling in training
total_tr = sum(class_counts)
class_weights = {i: total_tr / (NUM_CLASSES * c) for i, c in enumerate(class_counts)}
print('\nClass weights:')
for i, (key, w) in enumerate(zip(LABELS_MAP, class_weights.values())):
    print(f'  [{i}] {key:<30}  weight={w:.3f}')

split_info['class_weights'] = {str(k): float(v) for k, v in class_weights.items()}
split_info['class_counts'] = class_counts
with open('/content/model/split_info.json', 'w') as f:
    json.dump(split_info, f, indent=2)

print('\n✅ Notebook 1 complete! Proceed to 02_train_model.ipynb')